<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_13_2_anomaly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>

# T81-558: Anwendungen tiefer neuronaler Netzwerke
**Modul 14: Andere neuronale Netzwerktechniken**
* Dozent: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Jeff Heaton](https://sites.wustl.edu/jeffheaton/)
* Weitere Informationen finden Sie unter [class website](https://sites.wustl.edu/jeffheaton/t81-558/).

# Modul 13 Videomaterial

* Teil 13.1: Verwenden von Denoising Autocodern [[Video]](https://www.youtube.com/watch?v=BBrRD89sTk8&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi) [[Video]](https://www.youtube.com/watch?v=BBrRD89sTk8&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi)
* **Teil 13.2: Anomalieerkennung** [[Video]](https://www.youtube.com/watch?v=wubZ516TkI8&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi) [[Video]](https://www.youtube.com/watch?v=wubZ516TkI8&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi)
* Teil 13.3: Modelldrift und Neutraining [[Video]](https://www.youtube.com/watch?v=F4395B1ySpg&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi) [[Video]](https://www.youtube.com/watch?v=F4395B1ySpg&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi)
* Teil 13.4: Tensor Processing Units (TPUs) [[Video]](https://www.youtube.com/watch?v=Cp3xOyxOZNo&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi) [[Video]](https://www.youtube.com/watch?v=Cp3xOyxOZNo&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi)
* Teil 13.5: Zukünftige Richtungen in der künstlichen Intelligenz [[Video]](https://www.youtube.com/watch?v=RjxvEZh73Yc&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi) [[Video]](https://www.youtube.com/watch?v=RjxvEZh73Yc&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi)



# Google CoLab-Anweisungen

Der folgende Code überprüft, ob Google CoLab installiert ist, und richtet die richtigen Hardwareeinstellungen für PyTorch ein.


In [1]:
import torch

try:
    import google.colab

    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Nutzen Sie eine GPU oder MPS (Apple), sofern verfügbar. (siehe Modul 3.2)
import torch

has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Verwendetes Gerät: {device}")

Note: not using Google CoLab
Using device: mps


# Teil 13.2: Anomalieerkennung

Die Anomalieerkennung ist eine unbeaufsichtigte Trainingstechnik, die analysiert, inwieweit eingehende Daten von den Daten abweichen, die Sie zum Trainieren des neuronalen Netzwerks verwendet haben. Traditionell haben Cybersicherheitsexperten die Anomalieerkennung verwendet, um die Netzwerksicherheit zu gewährleisten. Sie können Anomalien jedoch in der Datenwissenschaft verwenden, um Eingaben zu erkennen, für die Sie Ihr neuronales Netzwerk nicht trainiert haben.

Es gibt mehrere Datensätze, die häufig zur Demonstration der Anomalieerkennung verwendet werden. In diesem Teil werden wir uns den KDD-99-Datensatz ansehen.


* [Stratosphere IPS Dataset](https://www.stratosphereips.org/category/dataset.html)
* [The ADFA Intrusion Detection Datasets (2013) – für HIDS](https://www.unsw.adfa.edu.au/unsw-canberra-cyber/cybersecurity/ADFA-IDS-Datasets/)
* [ITOC CDX (2009)](https://westpoint.edu/centers-and-research/cyber-research-center/data-sets)
* [KDD-99 Dataset](http://kdd.ics.uci.edu/databases/kddcup99/kddcup99.html)

## KDD99-Datensatz einlesen

Obwohl der KDD99-Datensatz über 20 Jahre alt ist, wird er immer noch häufig zur Demonstration von Intrusion Detection Systems (IDS) und Anomalieerkennung verwendet. KDD99 ist der Datensatz, der für den dritten internationalen Wettbewerb für Knowledge Discovery and Data Mining Tools verwendet wurde, der in Verbindung mit KDD-99, der fünften internationalen Konferenz für Knowledge Discovery und Data Mining, abgehalten wurde. Die Wettbewerbsaufgabe bestand darin, einen Netzwerk-Intrusion-Detector zu erstellen, ein Vorhersagemodell, das in der Lage ist, zwischen „schlechten“ Verbindungen, sogenannten Intrusionen oder Angriffen, und „guten“ normalen Verbindungen zu unterscheiden. Diese Datenbank enthält einen Standardsatz von zu prüfenden Daten, darunter verschiedene Intrusionen, die in einer militärischen Netzwerkumgebung simuliert wurden.

Der folgende Code liest den KDD99-CSV-Datensatz in einen Pandas-Datenrahmen ein. Das Standardformat von KDD99 enthält keine Spaltennamen. Aus diesem Grund fügt das Programm sie hinzu.

In [2]:
import pandas as pd
import urllib.request
import os

# Festlegen der Pandas-Anzeigeoptionen
set_option("display.max_columns", 6)
set_option("display.max_rows", 5)

# Laden Sie die Datei mit urllib herunter
url = "https://github.com/jeffheaton/jheaton-ds2/raw/main/kdd-with-columns.csv"
filename = "kdd-with-columns.csv"

if not os.path.isfile(filename):
    try:
        urllib.request.urlretrieve(url, filename)
    except:
        print("Error downloading")
        raise

print(filename)

# Originaldatei: http://kdd.ics.uci.edu/databases/kddcup99/kddcup99.html
df = read_csv(filename)

print("Read {} rows.".format(len(df)))
# df = df.sample(frac=0.1, replace=False) # Entfernen Sie das Kommentarzeichen aus dieser Zeile, um nur 10 % des Datensatzes zu erfassen
df.dropna(inplace=True, axis=1)
# Lassen Sie vorerst nur NAs weg (Zeilen mit fehlenden Werten).

# 5 Zeilen anzeigen
set_option("display.max_columns", 5)
set_option("display.max_rows", 5)
print(df)

kdd-with-columns.csv
Read 494021 rows.
        duration protocol_type  ... dst_host_srv_rerror_rate  outcome
0              0           tcp  ...                      0.0  normal.
1              0           tcp  ...                      0.0  normal.
...          ...           ...  ...                      ...      ...
494019         0           tcp  ...                      0.0  normal.
494020         0           tcp  ...                      0.0  normal.

[494021 rows x 42 columns]


Der KDD99-Datensatz enthält viele Spalten, die den Netzwerkstatus über Zeitintervalle definieren, in denen ein Cyberangriff stattgefunden haben könnte. Die Spalte „Ergebnis“ gibt entweder „normal“ an, was bedeutet, dass kein Angriff stattgefunden hat, oder die Art des durchgeführten Angriffs. Der folgende Code zeigt die Anzahl für jede Angriffsart und „normal“ an.

In [3]:
df.groupby("outcome")["outcome"].count()

outcome
back.               2203
buffer_overflow.      30
                    ... 
warezclient.        1020
warezmaster.          20
Name: outcome, Length: 23, dtype: int64

## Vorverarbeitung

Bevor wir die KDD99-Daten in das neuronale Netzwerk einspeisen können, müssen wir einige Vorverarbeitungsvorgänge durchführen. Zur Unterstützung der Vorverarbeitung stellen wir die folgenden beiden Funktionen zur Verfügung. Die erste Funktion konvertiert numerische Spalten in Z-Scores. Die zweite Funktion ersetzt kategorische Werte durch Dummy-Variablen.

In [4]:
import pandas as pd


def encode_numeric_zscore(df, name):
    """
    Apply z-score normalization to a specified numeric column.

    Parameters:
    df (DataFrame): The pandas DataFrame containing the column.
    name (str): The name of the column to normalize.
    """
    mean = df[name].mean()
    sd = df[name].std()
    df[name] = (df[name] - mean) / sd


def encode_text_dummy(df, name):
    """
    Convert a categorical column to dummy variables.

    Parameters:
    df (DataFrame): The pandas DataFrame containing the column.
    name (str): The name of the categorical column.
    """
    dummies = get_dummies(df[name], prefix=name, dtype=float)
    df = concat([df, dummies], axis=1)
    df.drop(name, axis=1, inplace=True)
    return df


def process_dataframe(df):
    """
    Process a DataFrame by encoding its features.

    Parameters:
    df (DataFrame): The pandas DataFrame to process.
    """
    for name in df.columns:
        if name == "outcome":
            continue
        # elif df[name].dtype == bool:
        # drucken("**", Name)
        # df[Name] = df[Name].astype(Float)
        elif name in [
            "protocol_type",
            "service",
            "flag",
            "land",
            "logged_in",
            "is_host_login",
            "is_guest_login",
        ]:
            df = encode_text_dummy(df, name)
        else:
            encode_numeric_zscore(df, name)
    return df

Dieser Code konvertiert alle numerischen Spalten in Z-Scores und alle Textspalten in Dummyvariablen. Wir verwenden diese Funktionen nun, um jede der Spalten vorzuverarbeiten. Sobald das Programm die Daten vorverarbeitet hat, zeigen wir die Ergebnisse an.

In [5]:
set_option("display.max_columns", 6)
set_option("display.max_rows", 5)

df = process_dataframe(df)
df.dropna(inplace=True, axis=1)
print(df.head())

   duration  src_bytes  dst_bytes  ...  is_host_login_0  is_guest_login_0  \
0 -0.067792  -0.002879   0.138664  ...              1.0               1.0   
1 -0.067792  -0.002820  -0.011578  ...              1.0               1.0   
2 -0.067792  -0.002824   0.014179  ...              1.0               1.0   
3 -0.067792  -0.002840   0.014179  ...              1.0               1.0   
4 -0.067792  -0.002842   0.035214  ...              1.0               1.0   

   is_guest_login_1  
0               0.0  
1               0.0  
2               0.0  
3               0.0  
4               0.0  

[5 rows x 121 columns]


Wir unterteilen die Daten in zwei Gruppen: „normal“ und die verschiedenen Angriffe, um eine Anomalieerkennung durchzuführen. Der folgende Code unterteilt die Daten in zwei Datenrahmen und zeigt die Größe jeder dieser beiden Gruppen an.

In [6]:
normal_mask = df["outcome"] == "normal."
attack_mask = df["outcome"] != "normal."

df.drop("outcome", axis=1, inplace=True)

df_normal = df[normal_mask]
df_attack = df[attack_mask]

print(f"Normale Anzahl: {len(df_normal)}")
print(f"Anzahl der Angriffe: {len(df_attack)}")

Normal count: 97278
Attack count: 396743


Als nächstes konvertieren wir diese beiden Datenrahmen in Numpy-Arrays. Keras benötigt dieses Format für Daten.

In [7]:
# Dies ist der numerische Merkmalsvektor, wie er an das neuronale Netz geht
x_normal = df_normal.values
x_attack = df_attack.values

## Den Autocoder trainieren

Es ist wichtig zu beachten, dass wir die Ergebnisspalte nicht als Bezeichnung für die Vorhersage verwenden. Wir werden einen Autocoder anhand der normalen Daten trainieren und sehen, wie gut er erkennen kann, dass die nicht als „normal“ gekennzeichneten Daten eine Anomalie darstellen. Diese Anomalieerkennung erfolgt unbeaufsichtigt; es gibt keinen Zielwert (y) zur Vorhersage.

Als nächstes teilen wir die normalen Daten in einen 25%-Testsatz und einen 75%-Trainingssatz auf. Das Programm wird die Testdaten verwenden, um ein frühzeitiges Stoppen zu ermöglichen.

In [8]:
from sklearn.model_selection import train_test_split

x_normal_train, x_normal_test = train_test_split(
    x_normal, test_size=0.25, random_state=42
)

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Konvertieren Sie Numpy-Arrays in PyTorch-Tensoren und verschieben Sie sie auf das entsprechende Gerät
x_normal_train_tensor = torch.tensor(x_normal_train).float().to(device)
x_normal_tensor = torch.tensor(x_normal).float().to(device)
x_attack_tensor = torch.tensor(x_attack).float().to(device)

# Erstellen Sie einen DataLoader für die Stapelverarbeitung
train_data = TensorDataset(x_normal_train_tensor, x_normal_train_tensor)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

# Definieren Sie das Modell mit Sequential
model = nn.Sequential(
    nn.Linear(x_normal.shape[1], 25),
    nn.ReLU(),
    nn.Linear(25, 3),
    nn.ReLU(),
    nn.Linear(3, 25),
    nn.ReLU(),
    nn.Linear(25, x_normal.shape[1])
).to(device)

# Verlust und Optimierer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10
# Trainingsschleife
for epoch in range(num_epochs):
    running_loss = 0
    den = 0
    for data in train_loader:
        inputs, targets = data
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss +=loss.item()
        den+=1

print(f"Epoche [{epoch + 1}/{num_epochs}], Verlust: {running_loss/den}")
    running_loss = 0.0


Epoch [1/10], Loss: 0.32732871251074563
Epoch [2/10], Loss: 0.2666971355484668
Epoch [3/10], Loss: 0.25296189310355927
Epoch [4/10], Loss: 0.2429972947542474
Epoch [5/10], Loss: 0.22758804882525285
Epoch [6/10], Loss: 0.2041895669215081
Epoch [7/10], Loss: 0.18810490698149232
Epoch [8/10], Loss: 0.1733971669941144
Epoch [9/10], Loss: 0.16838515430255874
Epoch [10/10], Loss: 0.1590980681635668


Wir zeigen die Größe der Trainings- und Testsätze an.

In [10]:
print(f"Normale Zuganzahl: {len(x_normal_train)}")
print(f"Anzahl der Normaltests: {len(x_normal_test)}")

Normal train count: 72958
Normal test count: 24320


Wir sind nun bereit, den Autocoder mit den normalen Daten zu trainieren. Der Autocoder wird lernen, die Daten auf einen Vektor mit nur drei Zahlen zu komprimieren. Der Autocoder sollte auch in der Lage sein, mit angemessener Genauigkeit zu dekomprimieren. Wie für Autocoder üblich, trainieren wir lediglich das neuronale Netzwerk, um dieselben Ausgabewerte zu erzeugen, die in die Eingabeebene eingespeist wurden.

## Erkennen einer Anomalie

Jetzt können wir sehen, ob die abnormalen Daten eine Anomalie darstellen. Die ersten beiden Werte zeigen die RMSE-Fehler innerhalb und außerhalb der Stichprobe. Diese beiden Werte sind mit etwa 0,33 relativ niedrig, da sie aus normalen Daten resultieren. Der viel höhere Fehler von 0,76 ist auf die abnormalen Daten zurückzuführen. Der Autocoder ist nicht so gut in der Lage, Daten zu kodieren, die einen Angriff darstellen. Dieser höhere Fehler weist auf eine Anomalie hin.

In [11]:
model.eval()  # Setzen Sie das Modell in den Auswertungsmodus


# Funktion zur Berechnung des RMSE
def calculate_rmse(model, data):
    with torch.no_grad():
        predictions = model(data)
        mse_loss = nn.MSELoss()(predictions, data)
    return torch.sqrt(mse_loss).item()


# Bewertung des Modells
score1 = calculate_rmse(model, torch.tensor(x_normal_test).float().to(device))
score2 = calculate_rmse(model, x_normal_tensor)
score3 = calculate_rmse(model, x_attack_tensor)

print(f"Außerhalb der Stichprobe normaler Wert (RMSE): {score1}")
print(f"Insample-Normalwert (RMSE): {score2}")
print(f"Punktzahl für laufenden Angriff (RMSE): {score3}")

Out of Sample Normal Score (RMSE): 0.391261488199234
Insample Normal Score (RMSE): 0.3845501244068146
Attack Underway Score (RMSE): 0.5126159191131592
